# 02 · Inventory Demand Forecast

Calculates per-SKU reorder quantities using a Weighted Moving Average (WMA) built on four rolling 90-day quarters.

**No visuals.** Output is a reorder quantity and status flag per SKU — an operational number intended for purchase-order generation, not exploration.

```
inventory-analysis/
├── notebooks/  02_inventory_forecast.ipynb   ← you are here
├── src/        inventory_forecast.py          ← pipeline
└── outputs/    Demand_Forecast_YYYYMMDD.xlsx
```

## Method

| Step | What | Why |
|------|------|-----|
| Filter | FAMILY sales only | CONTRACT bulk orders distort per-unit demand |
| Quarters | Four rolling 90-day windows | Captures recent demand shift without full-year lag |
| Blended ADS | α × selling_days + (1−α) × 90 | Prevents demand inflation for intermittent SKUs |
| Dynamic weights | Proportional to quarterly sales share | Auto-adapts to seasonality |
| Standard weights | 40 / 30 / 20 / 10 | Fallback for sparse SKUs (< 30 active days) |
| WMA | Σ(ADS_q × W_q) | Weighted by recency and seasonal activity |
| Stockout adj. | × (avail + 0.5 × missed) / avail | Recovers suppressed demand; capped at 1.5× |
| Reorder qty | max(0, Target(7M) − Effective_Stock) | Conservative: only orders what is genuinely short |


## 0 · Dependencies

In [ ]:
!pip install pandas numpy openpyxl -q

## 1 · Imports

In [9]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import inventory_forecast as inv
print('imports ok')

imports ok


## 2 · Load data

In [10]:
combined_df = pd.read_parquet(PROJECT_ROOT / 'outputs' / 'combined_df.parquet')
print(f'shape     : {combined_df.shape}')
print(f'range     : {combined_df["Date"].min().date()} → {combined_df["Date"].max().date()}')
print(f'columns   : {combined_df.columns.tolist()}')

shape     : (4222805, 40)
range     : 2019-01-01 → 2025-12-20
columns   : ['Section', 'Section Name', 'bal Value', 'bal Qty', 'U Price', 'Client', 'Outlet', 'Outlet Name', 'Date', 'SKU', 'Catalog No.', 'Year', 'Total_Cost', 'Total_Profit', 'Profit_Margin_%', 'CURRENT_STOCK', 'OUTSTANDING', 'PR', 'EFFECTIVE_STOCK', 'CATEGORY', 'Supplier', 'LAST_ENTRY_DATE', 'Nb. Days (Avail. Balance)', 'COST_PER_DOLLAR', 'COLOR_NAME', 'ARTPATERN', 'TEXTURE', 'SUB_CATEGORY', 'IS_PLAIN_SECTION', 'CATALOG_NO', 'COLLECTION_NAME', 'SERIAL', 'SKU-STATUES-Mahmoud', 'SKU STATUES -Python', 'OLD_QTY', 'First_Inv_Date', 'Outlet_Type', 'Area_Master_Category', 'Country', 'Sales_Type']


### Stockout reference file (optional)

Load `nb.xlsx` if available. Contains `Nb. Days (Avail. Balance)` per SKU — number of days the SKU had stock in the last 120 days. Used to detect and correct demand suppression from stockouts.

If the file is not present the forecast runs without stockout adjustment.

In [11]:
nb_path = PROJECT_ROOT / 'data' / 'nb.xlsx'

if nb_path.exists():
    nb_days_df = pd.read_excel(nb_path)
    nb_days_df['SKU'] = nb_days_df['SKU'].astype(str).str.strip()
    print(f'nb_days_df loaded  ·  {len(nb_days_df):,} rows')
else:
    nb_days_df = None
    print('nb.xlsx not found — stockout adjustment disabled')

nb.xlsx not found — stockout adjustment disabled


In [12]:
combined_df.columns

Index(['Section', 'Section Name', 'bal Value', 'bal Qty', 'U Price', 'Client',
       'Outlet', 'Outlet Name', 'Date', 'SKU', 'Catalog No.', 'Year',
       'Total_Cost', 'Total_Profit', 'Profit_Margin_%', 'CURRENT_STOCK',
       'OUTSTANDING', 'PR', 'EFFECTIVE_STOCK', 'CATEGORY', 'Supplier',
       'LAST_ENTRY_DATE', 'Nb. Days (Avail. Balance)', 'COST_PER_DOLLAR',
       'COLOR_NAME', 'ARTPATERN', 'TEXTURE', 'SUB_CATEGORY',
       'IS_PLAIN_SECTION', 'CATALOG_NO', 'COLLECTION_NAME', 'SERIAL',
       'SKU-STATUES-Mahmoud', 'SKU STATUES -Python', 'OLD_QTY',
       'First_Inv_Date', 'Outlet_Type', 'Area_Master_Category', 'Country',
       'Sales_Type'],
      dtype='str')

## 3 · Run forecast

In [13]:
forecast_df, output_path = inv.run_demand_forecast(
    combined_df,
    nb_days_df=nb_days_df,
    analysis_date=None,       # defaults to today
    output_dir=str(PROJECT_ROOT / 'outputs'),
)


── Demand Forecast Pipeline  ·  2026-03-15 ──────────────────
  FAMILY filter : 4,193,305 / 4,222,805 records
  Date range    : 2025-03-16 → 2025-12-20
  SKUs          : 27,848
  Avg coverage  : 1.5%

  Stage 1 · Quarterly metrics
  Stage 2 · Dynamic weights
    dynamic  : 730  |  standard : 27,118
  Stage 3 · WMA
    WMA used : 0  |  actual avg : 27,848
  Stage 4 · Stockout adjustment
    adjusted : 0 SKUs
  Stage 5 · Inventory positions

── Results ───────────────────────────────────────────────────
  reorder required :    2,271
  total reorder qty:   52,593
  critical         :    2,282
  out of stock     :    2,076
  excess stock     :   23,273

  Stage 6 · Saving Excel
  saved  ·  Demand_Forecast_20260315_014403.xlsx


## 4 · Validation

These checks run after every forecast to confirm the output is internally consistent before it is used to generate purchase orders.

### 4.1 · Status distribution

In [14]:
from IPython.display import display

status = (
    forecast_df['STATUS']
    .value_counts()
    .reset_index()
)
status.columns = ['STATUS', 'Count']
status['% of SKUs'] = (status['Count'] / len(forecast_df) * 100).round(1)
display(
    status.style
    .format({'Count': '{:,}', '% of SKUs': '{:.1f}%'})
    .bar(subset=['Count'], color='#1f77b4')
)

,STATUS,Count,% of SKUs
0,EXCESS,"23,273",83.6%
1,CRITICAL,"2,282",8.2%
2,OUT_OF_STOCK,"2,076",7.5%
3,REORDER,207,0.7%
4,OK,10,0.0%


### 4.2 · Forecast method split

In [15]:
method = (
    forecast_df['Forecast_Method']
    .value_counts()
    .reset_index()
)
method.columns = ['Method', 'Count']
method['% of SKUs'] = (method['Count'] / len(forecast_df) * 100).round(1)

weight_method = (
    forecast_df['Weight_Method']
    .value_counts()
    .reset_index()
)
weight_method.columns = ['Weight_Method', 'Count']
weight_method['% of SKUs'] = (weight_method['Count'] / len(forecast_df) * 100).round(1)

print('Forecast method:')
display(method.style.format({'Count': '{:,}', '% of SKUs': '{:.1f}%'}))
print('Weight method:')
display(weight_method.style.format({'Count': '{:,}', '% of SKUs': '{:.1f}%'}))

Forecast method:


,Method,Count,% of SKUs
0,Actual (Conservative),"27,848",100.0%


Weight method:


,Weight_Method,Count,% of SKUs
0,Standard,"27,118",97.4%
1,Dynamic,730,2.6%


### 4.3 · Top reorder SKUs

In [16]:
reorder_cols = [
    'SKU', 'CATEGORY',
    'STATUS', 'CURRENT_STOCK', 'Effective_Stock',
    'Monthly_Demand_Final', 'Target_Stock_7M', 'Reorder_Qty',
    'Days_Coverage', 'Weight_Method', 'Forecast_Method',
    'Stockout_Adjusted',
]
if 'Section' in forecast_df.columns:
    reorder_cols.insert(1, 'Section')

present = [c for c in reorder_cols if c in forecast_df.columns]
top_reorder = (
    forecast_df[forecast_df['Reorder_Qty'] > 0][present]
    .sort_values('Reorder_Qty', ascending=False)
    .head(30)
)

fmt = {
    'Monthly_Demand_Final': '{:.1f}',
    'Target_Stock_7M':      '{:,}',
    'Reorder_Qty':          '{:,}',
    'Effective_Stock':      '{:,}',
    'CURRENT_STOCK':        '{:,}',
    'Days_Coverage':        '{:.0f}',
}
fmt_present = {k: v for k, v in fmt.items() if k in present}

print(f'Reorder required: {(forecast_df["Reorder_Qty"] > 0).sum():,} SKUs  |  '
      f'Total qty: {forecast_df["Reorder_Qty"].sum():,.0f}')

display(
    top_reorder.style
    .format(fmt_present)
    .background_gradient(subset=['Reorder_Qty'], cmap='Reds')
    .bar(subset=['Days_Coverage'] if 'Days_Coverage' in present else [], color='#aec7e8')
)

Reorder required: 2,271 SKUs  |  Total qty: 52,593


,SKU,Section,CATEGORY,STATUS,CURRENT_STOCK,Effective_Stock,Monthly_Demand_Final,Target_Stock_7M,Reorder_Qty,Days_Coverage,Weight_Method,Forecast_Method,Stockout_Adjusted
22377,5068010011,5068,C,REORDER,33.0,33,91.8,642,609,11,Dynamic,Actual (Conservative),No
22376,5068010010,5068,C,CRITICAL,17.0,17,87.5,612,595,6,Dynamic,Actual (Conservative),No
27731,5365010011,5365,B,REORDER,471.0,471,151.0,"1,057",586,94,Dynamic,Actual (Conservative),No
22378,5068010012,5068,C,REORDER,50.0,50,90.3,631,581,17,Dynamic,Actual (Conservative),No
25060,5146010015,5146,C,CRITICAL,6.0,6,58.8,411,405,3,Dynamic,Actual (Conservative),No
27749,5368010010,5368,B,REORDER,100.0,100,71.1,497,397,42,Dynamic,Actual (Conservative),No
25831,5189011012,5189,C,REORDER,298.0,298,97.8,684,386,91,Dynamic,Actual (Conservative),No
24550,5125010013,5125,C,REORDER,93.0,93,61.8,432,339,45,Dynamic,Actual (Conservative),No
24557,5125010020,5125,C,CRITICAL,18.0,18,50.8,355,337,11,Standard,Actual (Conservative),No
25507,5183010017,5183,C,REORDER,71.0,71,57.0,399,328,37,Standard,Actual (Conservative),No


### 4.4 · Section-level reorder summary

In [17]:
if 'Section' in forecast_df.columns:
    sec = (
        forecast_df
        .groupby('Section')
        .agg(
            SKUs              = ('SKU',               'count'),
            Total_Sales_1Y    = ('Total_Sales_1Y',     'sum'),
            Reorder_Qty       = ('Reorder_Qty',        'sum'),
            Effective_Stock   = ('Effective_Stock',    'sum'),
            Reorder_SKUs      = ('Reorder_Qty',        lambda x: (x > 0).sum()),
            Critical_SKUs     = ('STATUS',             lambda x: x.isin(['CRITICAL','OUT_OF_STOCK']).sum()),
        )
        .reset_index()
        .sort_values('Reorder_Qty', ascending=False)
    )
    sec['Reorder_%'] = (sec['Reorder_SKUs'] / sec['SKUs'] * 100).round(1)

    display(
        sec.head(20).style
        .format({
            'Total_Sales_1Y':  '{:,.0f}',
            'Reorder_Qty':     '{:,}',
            'Effective_Stock': '{:,}',
            'Reorder_%':       '{:.1f}%',
        })
        .background_gradient(subset=['Reorder_Qty'], cmap='Oranges')
    )
else:
    print('Section column not available.')

,Section,SKUs,Total_Sales_1Y,Reorder_Qty,Effective_Stock,Reorder_SKUs,Critical_SKUs,Reorder_%
1671,5068,52,"11,200","2,300","49,282",8,6,15.4%
1613,4999,86,"8,152","1,743","56,452",24,23,27.9%
1614,5000,66,"7,685","1,321","54,800",15,15,22.7%
1711,5125,87,"17,522","1,177","161,124",6,6,6.9%
1330,4689,151,"30,042",942,"137,496",7,3,4.6%
261,3190,11,"1,919",780,517,7,3,63.6%
1486,4858,27,"3,079",777,"17,036",8,6,29.6%
1658,5054,82,"5,924",696,"45,454",9,11,11.0%
1710,5124,86,"5,377",677,"87,517",6,6,7.0%
1869,5365,7,"3,354",586,"9,371",1,0,14.3%


### 4.5 · Sanity checks

In [18]:
checks = {
    'Negative Reorder_Qty'       : (forecast_df['Reorder_Qty'] < 0).sum(),
    'Zero Monthly_Demand_Final'  : (forecast_df['Monthly_Demand_Final'] == 0).sum(),
    'Coverage > 365 days'        : (forecast_df['Days_Coverage'] > 365).sum(),
    'Missing CATEGORY'           : forecast_df['CATEGORY'].isna().sum(),
    'Stockout adjusted'          : (forecast_df['Stockout_Adjusted'] == 'Yes').sum(),
}
result = pd.DataFrame(list(checks.items()), columns=['Check', 'Count'])
result['Pass'] = result['Count'].apply(
    lambda x: '✓' if x == 0 else '⚠')
# Stockout adjusted is informational, not a failure
result.loc[result['Check'] == 'Stockout adjusted', 'Pass'] = 'ℹ'
display(result.style.apply(
    lambda col: ['color: green' if v == '✓' else
                 'color: grey'  if v == 'ℹ' else
                 'color: orange' for v in col]
    if col.name == 'Pass' else [''] * len(col), axis=0
))

,Check,Count,Pass
0,Negative Reorder_Qty,0,✓
1,Zero Monthly_Demand_Final,167,⚠
2,Coverage > 365 days,24329,⚠
3,Missing CATEGORY,55,⚠
4,Stockout adjusted,0,ℹ


## 5 · Output file

The Excel file contains four sheets:

| Sheet | Contents |
|-------|----------|
| Summary | Run metadata and key totals |
| All SKUs | Full forecast table — one row per SKU |
| Reorder Required | Only SKUs with Reorder_Qty > 0, sorted descending |
| Critical | SKUs with STATUS = CRITICAL or OUT_OF_STOCK, sorted by days coverage |
| Section Summary | Aggregated reorder quantities by section |


In [19]:
print(f'Output : {output_path}')
print(f'Size   : {output_path.stat().st_size / 1024:.1f} KB')

Output : D:\Projects\InventoryDeepDive\outputs\Demand_Forecast_20260315_014403.xlsx
Size   : 7197.7 KB
